Restarted .venv (Python 3.10.16)

In [3]:
from sentence_transformers import SentenceTransformer, models
import torch

word_embedding_model = models.Transformer(
    'emilyalsentzer/Bio_ClinicalBERT',
    max_seq_length=512,
    model_args={"torch_dtype": torch.float32} 
)

pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_cls_token=False,
    pooling_mode_mean_tokens=True,
    pooling_mode_max_tokens=False
)

model = SentenceTransformer(modules=[word_embedding_model, pooling_model], device='mps')

RuntimeError: Failed to import transformers.modeling_utils because of the following error (look up to see its traceback):
partially initialized module 'torchvision' has no attribute 'extension' (most likely due to a circular import)

In [ ]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_pickle('data/preprocess_clean_remove_rare.pkl')

In [ ]:
df.head(2)

,Unnamed: 0,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,PTEXT,FTEXT
0,0,316398,31608,152365.0,2133-01-19,2133-01-19 06:53:00,2133-01-19 06:53:16,Nursing,Nursing Progress Note,19650.0,NaN,"TITLE:\n Respiratory failure, acute (not ARD...","[titl, respiratori, failur, acut, doctor, last...","[titl, respiratori, failur, acut, doctor, last..."
1,1,315910,31608,152365.0,2133-01-13,2133-01-13 06:21:00,2133-01-13 06:21:26,Nursing,Nursing Progress Note,21198.0,NaN,Ineffective Coping\n Assessment:\n Pt is n...,"[ineffect, cope, assess, pt, dnr, pt, ask, res...","[ineffect, cope, assess, pt, dnr, pt, ask, res..."


In [ ]:
df['SENT'] = df['FTEXT'].apply(lambda tokens: ' '.join(tokens))

In [ ]:
df.head(2)

,Unnamed: 0,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,PTEXT,FTEXT,SENT
0,0,316398,31608,152365.0,2133-01-19,2133-01-19 06:53:00,2133-01-19 06:53:16,Nursing,Nursing Progress Note,19650.0,NaN,"TITLE:\n Respiratory failure, acute (not ARD...","[titl, respiratori, failur, acut, doctor, last...","[titl, respiratori, failur, acut, doctor, last...",titl respiratori failur acut doctor last name ...
1,1,315910,31608,152365.0,2133-01-13,2133-01-13 06:21:00,2133-01-13 06:21:26,Nursing,Nursing Progress Note,21198.0,NaN,Ineffective Coping\n Assessment:\n Pt is n...,"[ineffect, cope, assess, pt, dnr, pt, ask, res...","[ineffect, cope, assess, pt, dnr, pt, ask, res...",ineffect cope assess pt dnr pt ask resp tech c...


In [ ]:
df['SENT'][5]

'respiratori failur acut doctor last name assess action respons plan ineffect cope assess action respons plan'

In [ ]:
sentences = df['SENT'].tolist()

In [ ]:
df.shape

(142110, 15)

In [ ]:
len(sentences)

142110

In [ ]:
embeddings = model.encode(sentences, batch_size=16, show_progress_bar=True)

Batches:   0%|          | 0/8882 [00:00<?, ?it/s]

In [ ]:
import pickle

with open("embeddings_MEAN.pkl", "wb") as f:
    pickle.dump(embeddings, f)

In [ ]:
type(embeddings)

numpy.ndarray

In [ ]:
embeddings

array([[-0.03600806, -0.05763245, -0.5764836 , ...,  0.05967164,
        -0.06377713, -0.15175182],
       [ 0.04741407, -0.25133044, -0.42309144, ...,  0.02087078,
         0.12380733, -0.0428645 ],
       [ 0.0549481 ,  0.30195963, -0.5842947 , ..., -0.05654861,
         0.12264945, -0.35312414],
       ...,
       [ 0.01555623,  0.24602246, -0.39466158, ..., -0.06074153,
        -0.00849961, -0.11809866],
       [ 0.07003576, -0.01468849, -0.46918246, ...,  0.03723217,
         0.13071527, -0.04235954],
       [ 0.06021408,  0.03543237, -0.64930576, ...,  0.24909644,
         0.01986872, -0.04088048]], dtype=float32)

In [ ]:
df['EMBEDDING'] = list(embeddings)

In [ ]:
df.head()

,Unnamed: 0,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,PTEXT,FTEXT,SENT,EMBEDDING
0,0,316398,31608,152365.0,2133-01-19,2133-01-19 06:53:00,2133-01-19 06:53:16,Nursing,Nursing Progress Note,19650.0,NaN,"TITLE:\n Respiratory failure, acute (not ARD...","[titl, respiratori, failur, acut, doctor, last...","[titl, respiratori, failur, acut, doctor, last...",titl respiratori failur acut doctor last name ...,"[-0.036008064, -0.057632454, -0.5764836, 0.473..."
1,1,315910,31608,152365.0,2133-01-13,2133-01-13 06:21:00,2133-01-13 06:21:26,Nursing,Nursing Progress Note,21198.0,NaN,Ineffective Coping\n Assessment:\n Pt is n...,"[ineffect, cope, assess, pt, dnr, pt, ask, res...","[ineffect, cope, assess, pt, dnr, pt, ask, res...",ineffect cope assess pt dnr pt ask resp tech c...,"[0.04741407, -0.25133044, -0.42309144, 0.49156..."
2,2,316202,31608,152365.0,2133-01-16,2133-01-16 03:53:00,2133-01-16 03:53:57,Nursing,Nursing Progress Note,14442.0,NaN,"Respiratory failure, acute (not ARDS/[**Doctor...","[respiratori, failur, acut, doctor, last, name...","[respiratori, failur, acut, doctor, last, name...",respiratori failur acut doctor last name asses...,"[0.054948103, 0.30195963, -0.5842947, 0.443203..."
3,3,316288,31608,152365.0,2133-01-17,2133-01-17 15:10:00,2133-01-17 15:10:44,Nursing,Nursing Progress Note,15065.0,NaN,"Respiratory failure, acute (not ARDS/[**Doctor...","[respiratori, failur, acut, doctor, last, name...","[respiratori, failur, acut, doctor, last, name...",respiratori failur acut doctor last name asses...,"[0.14360966, -0.1609861, -0.49888694, 0.472669..."
4,5,316296,31608,152365.0,2133-01-17,2133-01-17 15:10:00,2133-01-17 18:08:06,Nursing,Nursing Progress Note,15065.0,NaN,"Respiratory failure, acute (not ARDS/[**Doctor...","[respiratori, failur, acut, doctor, last, name...","[respiratori, failur, acut, doctor, last, name...",respiratori failur acut doctor last name asses...,"[0.14360966, -0.1609861, -0.49888694, 0.472669..."


In [ ]:
df.to_pickle('data/notes_with_embeddings_MEAN.pkl')